# 单元09：学习曲线与K折交叉验证

状态：学习中，2026-10-29建立。本Notebook是逐步填写的教学工作簿，不是已完成实验。

材料情境为7085Al/ZrB2，所有新数据必须标注synthetic data。2026-10-30教师已按预设配方生成新的150条教学CSV；尚未划分数据或拟合模型，没有真实模型曲线、指标或正式评估结果。

课程依据：markdown/week6.md第10.3—10.6节；code/ex5-bias vs variance/。沿用单元08已经学过的线性回归、多项式和Ridge，本单元不引入其他模型。


## 实验预设（在数据生成和查看结果前固定）

本单元只使用全新的 **synthetic data（人工模拟数据）**，不复用单元08的CSV、训练/验证/测试分组或已知分数。以下数量和设置是在生成数据前确定的实验设计；2026-10-30仅完成教师生成CSV，尚未划分或执行模型，不把设计写成实际评价结果。

- 教学问题：单一输入 ZrB2_wt_pct → 模拟目标 UTS_MPa。wt.%数值1表示1 wt.%，UTS单位MPa；关系为数学教学设定，不是7085Al强化机制或实验标定。
- 新数据配方：NumPy default_rng(20261029)；先生成150个Uniform(0,2)输入，再生成150个Normal(0,6)噪声；模拟目标为465 + 60*x - 24*x**2 + noise。不按预期图形重抽样或更换种子。
- 以sample_id固定追溯行身份。原始输入X预期shape=(150,1)，目标y沿用二维列约定(150,1)；这些是计划shape，运行后另核验。
- 在任何拟合、选模、目标散点图或指标分析之前，用train_test_split(test_size=0.2, random_state=20261030)分离120条开发数据、30条独立保留测试数据。记录索引；保留测试不预览目标、作图、参与曲线或交叉验证。
- 先在开发数据内部按test_size=0.2、random_state=20261031建立固定96训练/24验证的入门学习曲线。训练子集顺序按固定划分返回顺序保留；样本量预设为12、24、48、72、96。每点重新训练完整Pipeline，训练误差仅在对应训练子集上计算，验证组固定。有限曲线无需严格单调。
- 然后在全部120条开发数据内部用KFold(n_splits=5, shuffle=True, random_state=20261032)，两个候选共用同一组折索引。每折计划96训练/24验证；只用开发数据，不把保留测试混入K-fold。
- 仅比较两个预设候选：StandardScaler → LinearRegression（一次线性回归）；PolynomialFeatures(degree=4, include_bias=False) → StandardScaler → Ridge(alpha=1.0)。次数和强度是在查看结果前给定的教学方案，不由单元08最优值迁移，不开展网格搜索或新模型。
- 每个训练子集和每个训练折都拟合自己的完整Pipeline。PolynomialFeatures没有学习均值/方差，但仍封装在折内；StandardScaler只学习本训练部分的均值/方差，验证只transform/predict。
- 评价使用不含正则化惩罚项的MAE（MPa）。使用cross_validate时明确scoring='neg_mean_absolute_error'并将负分转换成正MAE；实际键名、输入/输出与shape在代码教学时逐段解释。
- 报告5个验证折的MAE、算术平均值、描述性标准差（本单元预定np.std(ddof=0)），并声明标准差反映该固定划分下的折间波动，不是置信区间或所有未知材料的误差保证。训练折高度重叠，不把5折当成5次独立材料实验。
- 若依据开发CV选定一个方案，先冻结选择、流水线与最终评价步骤，才可用全部开发数据重拟合并一次评价本单元保留测试；不凭测试结果改次数、alpha、划分或种子。本阶段没有开放测试执行。
- learning_curve函数放在理解K折后再实现CV学习曲线；输入为完整Pipeline和原始开发数据，训练量指每折训练部分使用的样本量。函数输出的二维分数数组与fold维度由学生先判断。
- GroupKFold仅讲分组独立性原理；本轮不生成虚构炉次、不完成分组实践。真实炉次或文献来源需有实际元数据并匹配预测目标。

## 证据记录规则

每次代码教学先解释输入、输出、shape和来源。学生先写，教师先审查并提示，再由学生运行保存；没有真实输出就标“未运行”。教师辅助检查与学生运行分开记录；所有图像先由学生独立判断趋势、原因与限制，不预填诊断或候选排名。


## 概念入口Q0（历史；当前实践见下方）

单元08已正确说明横轴为训练样本数和增加数据的诊断用途，尚未实际画过学习曲线。这里检查新的实现步骤，不重学整章。

**教学设计，非实际运行结果：** 模型的多项式次数和Ridge的alpha都保持固定。要画训练样本数分别为12、24、48时的误差点：你会分别用对应数量的训练样本重新fit，还是先用全部训练样本fit一次，再换不同数量的样本计算误差？请说明理由。

Q0首次选第二种错误，已提示后纠正，完整原答与帮助层级保留在四类档案；本题不是单元末正式8题。

教师阶段记录（2026-10-30，不代填下方学生理解）：Q10首次答‘误差没有达到目标要求’，Q11首次答‘趋于平缓’，均为手工synthetic示意图B的正确观察；Q12首次答‘左侧12条，模型训练误差显著低于验证误差，为过拟合’，正确指出图A少样本端的过拟合常见信号与两种误差依据。图A/B都不是实际模型运行，不能仅凭一处误差差距确诊真实模型的原因；不是旧题独立复查或正式8题成绩。


## 我的理解

待学生作答；教师不预填。


## 实践第1步：读取新数据并先封存测试集（学生先写，尚未运行）

数据文件：[data/unit09-learning-curves-synthetic.csv](data/unit09-learning-curves-synthetic.csv)。由教师使用NumPy 2.2.6按上方预先固定的default_rng(20261029)配方生成，不重抽种子、不使用单元08数据。150条synthetic data，仅为数学教学设定，不是实验结果。

三列为sample_id、ZrB2_wt_pct、UTS_MPa。sample_id为U09-001至U09-150，只记录行身份，不作为模型特征。ZrB2数值1表示1 wt.%；UTS单位MPa。先生成全部150个Uniform(0,2)输入，再生成150个Normal(0,6)噪声，按465 + 60*x - 24*x**2 + noise构造目标。CSV浮点数以17位有效数字保存，已核验读取后无数值变化；这仅是数据文件检查，不是模型评价。

CSV SHA256：1486c2ffd5758f68fdd79b4d1cd3743e6e7aa6363d5e2dfcc43fbb67a468f165。当前仅完成生成，无数据划分、模型fit/predict/评分或学生Notebook执行。

请在下面已有代码格中完成：导入pandas和train_test_split；读取CSV为表格对象df09；保持CSV行顺序，用test_size=0.2、random_state=20261030一次划分为df09_dev和df09_holdout；只打印两者shape，不展示目标值、统计量或散点图。先在对话发代码检查，通过后再运行并保存。

输入df09预期shape=(150,3)，输出df09_dev预期(120,3)、df09_holdout预期(30,3)。这是计划数量，不是已运行输出。120条开发数据用于后续曲线/CV，30条holdout是预先保留的独立测试数据，本阶段只检查行列数量并封存。sample_id随整行一起划分，稍后保存划分ID。不要预先排序或根据UTS筛样本；不使用holdout选次数、alpha或随机种子。

固定两个候选、学习曲线循环、独立读图、5折轮换、Pipeline/cross_validate和评分汇总之后依次进行；本格不提供完整解法。


In [ ]:
# 概念解释通过后，由学生逐步编写本单元代码。
# 当前没有执行，也没有模拟模型分数。


## 结课评估（尚未开始）

完成教学与实践后逐题进行3道概念题、2道数学理解题、2道Python题、1道材料迁移题。首次错误不得由后续正确答案覆盖。正式题0/8作答，0/8验收；本Notebook仍为学习中。


## 来源与使用限制

- 仓库课程：markdown/week6.md 10.3—10.6；code/ex5-bias vs variance/ML-Exercise5.ipynb。
- 课程10.3的“交叉验证集”主要指固定验证集，不等同K-fold。10.3测试误差式误用了cv下标，按test样本解释。
- 原Ex5分别对X/Xval/Xtest计算缩放统计量，且末尾用测试误差选lambda；不把这些步骤作为本单元的验证范例。原文件保留不动。
- [scikit-learn学习曲线](https://scikit-learn.org/stable/modules/learning_curve.html)
- [交叉验证与分组划分](https://scikit-learn.org/stable/modules/cross_validation.html)
- [评分约定](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter)
- [Pipeline与数据泄漏](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

数据来自数学教学设定，任何CV分数均不能写成真实材料实验性能或全部未知材料的预测保证。
